# **Get CTA Alerts from Transit Chicago**

In [0]:
%sh
# Get the latest alerts from CTA
curl -o /Volumes/main/cta/cta_data/routes.aspx -O https://lapi.transitchicago.com/api/1.0/routes.aspx

# Let's explore the data that we downloaded

In [0]:
%sh head '/Volumes/main/cta/cta_data/routes.aspx'

# **Read the data into a dataframe and write it out the GIT folder**

In [0]:
from pyspark.sql.functions import col, explode

# 1. Read the XML with rowTag set to the root element (CTARoutes)
df = (spark.read
      .format("xml")
      .option("rowTag", "CTARoutes")
      .option("inferSchema", "true")
      .load("/Volumes/main/cta/cta_data/routes.aspx"))

# 2. Explode the RouteInfo array, keep TimeStamp, and select columns (skipping RouteURL)
df_final = ( df.select(
    col("TimeStamp"),
    explode(col("RouteInfo")).alias("route")
).select(
    col("TimeStamp"),
    col("route.Route"),
    col("route.RouteColorCode"),
    col("route.RouteTextColor"),
    col("route.ServiceId"),
    col("route.RouteStatus"),
    col("route.RouteStatusColor")
).where("RouteStatus != 'Normal Service'")
)

# Display the resulting dataframe
display(df_final)

# Convert your final dataframe to pandas so it's easy to dump into a small JSON file
pdf = df_final.toPandas()

# Define the target path inside your Databricks Git folder
git_folder_path = "/Workspace/Users/pravinboniface@yahoo.com/cta/alerts.json"

# Save the dataset as JSON
pdf.to_json(git_folder_path, orient="records", date_format="iso")
print(f"Alerts successfully written to Git folder: {git_folder_path}")


In [0]:
import os
import xml.etree.ElementTree as ET
import requests

# Set to True during development to read from disk.
# Set to False when deploying to query live CTA servers.
USE_LOCAL_FIXTURE = True
FIXTURE_PATH = "/Workspace/Users/pravinboniface@yahoo.com/routes.aspx"  # Changed from "routes.xml" to "routes.aspx"
CTA_ROUTES_URL = "http://lapi.transitchicago.com/api/1.0/routes.aspx"


def get_cta_raw_data() -> bytes:
    """Fetch CTA XML data either from a local fixture file or live network."""
    if USE_LOCAL_FIXTURE:
        if not os.path.exists(FIXTURE_PATH):
            raise FileNotFoundError(
                f"Local fixture '{FIXTURE_PATH}' not found. "
                f"Save a response file first or set USE_LOCAL_FIXTURE = False."
            )
        print(f"[Dev Mode] Reading data from local fixture: {FIXTURE_PATH}")
        with open(FIXTURE_PATH, "rb") as f:
            return f.read()
    else:
        print("[Prod Mode] Fetching live data from CTA API...")
        response = requests.get(CTA_ROUTES_URL, timeout=10)
        response.raise_for_status()
        return response.content


def parse_route_statuses(xml_bytes: bytes) -> dict:
    """Parse raw XML bytes into a structured dictionary."""
    statuses = {}
    root = ET.fromstring(xml_bytes)
    """First let us get the timestamp and error code"""
    ts = root.findtext("TimeStamp","N/A")
    print(f"{ts=}")
    error_code = root.findtext("ErrorCode","N/A")
    print(f"{error_code=}")

    for route in root.findall(".//RouteInfo"):
        status = route.findtext("RouteStatus", "Unknown")
        if status == "Unknown" or status == "Normal Service":
            continue
        route_id = route.findtext("Route", "N/A")
        RouteColorCode = route.findtext("RouteColorCode", "N/A")
        RouteTextColor = route.findtext("RouteTextColor", "N/A")
        RouteStatusColor = route.findtext("RouteStatusColor","N/A")

        statuses[route_id] = {
            "status": status
            ,"color": RouteColorCode
            ,"text_color": RouteTextColor
            ,"status_color": RouteStatusColor
        }
    return statuses


# Example Usage
if __name__ == "__main__":
    raw_data = get_cta_raw_data()
    routes = parse_route_statuses(raw_data)


    # 'c_type' is 38 for text color, 48 for background color
    ansi = lambda h, c_type=38: f"\033[{c_type};2;{int(h[0:2],16)};{int(h[2:4],16)};{int(h[4:6],16)}m" if len(h) >= 6 else ""
    RESET = "\033[0m"

    print(f"\nSuccessfully loaded {len(routes)} routes.")

    for r_id, info in list(routes.items())[:500]:
        # 48 is for background, 38 is for text foreground
        bg_color = ansi(info['color'], 48)
        tx_color = ansi(info['text_color'], 38)
        st_color = ansi(info['status_color'], 38)

        # Renders bullet in status color, and the Route Badge in its exact styling
        print(f"  {st_color}•{RESET} {bg_color}{tx_color}[{r_id}]{RESET} : {info['status']}")

